# One prompt, four asnwers

 "I have a raspberry Pi with a microphone attached to it. When it's windy, how can I minimise the noise created by the wind? Is there a way to measure wind noise? maybe create a python script for it?"

In [ ]:
#chatgpt 4o 
import numpy as np
import sounddevice as sd
import scipy.fftpack
import matplotlib.pyplot as plt

DURATION = 2  # seconds
SAMPLERATE = 44100

def record_audio():
    print("Recording...")
    audio = sd.rec(int(DURATION * SAMPLERATE), samplerate=SAMPLERATE, channels=1, dtype='float64')
    sd.wait()
    return audio.flatten()

def analyze_wind_noise(audio):
    fft_vals = np.abs(np.fft.rfft(audio))
    fft_freqs = np.fft.rfftfreq(len(audio), 1/SAMPLERATE)
    
    # Focus on low frequencies (<300 Hz)
    low_freq_energy = np.sum(fft_vals[(fft_freqs > 20) & (fft_freqs < 300)])
    total_energy = np.sum(fft_vals)

    ratio = low_freq_energy / total_energy
    print(f"Low-frequency energy ratio: {ratio:.2f}")

    # Threshold (tweak depending on your mic)
    if ratio > 0.3:
        print("⚠️ Likely wind noise detected!")
    else:
        print("✅ No significant wind noise detected.")

    # Optional: plot
    plt.plot(fft_freqs, fft_vals)
    plt.title("Frequency Spectrum")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.xlim(0, 1000)
    plt.show()

audio = record_audio()
analyze_wind_noise(audio)

In [ ]:
#lechat mistral
import numpy as np
import soundfile as sf
from scipy.signal import butter, filtfilt

def butter_lowpass(cutoff, fs, order=5):
    """Design a Butterworth low-pass filter."""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a

def apply_lowpass_filter(data, cutoff, fs, order=5):
    """Apply a low-pass filter to the data."""
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = filtfilt(b, a, data)
    return y

def reduce_wind_noise(input_file, output_file, cutoff_freq=300, order=5):
    """Reduce wind noise by applying a low-pass filter."""
    # Read the audio file
    data, samplerate = sf.read(input_file)

    # Apply low-pass filter
    filtered_data = apply_lowpass_filter(data, cutoff_freq, samplerate, order)

    # Write the filtered data to a new audio file
    sf.write(output_file, filtered_data, samplerate)

# Example usage
input_audio_file = 'windy_audio.wav'  # Replace with your input file
output_audio_file = 'filtered_audio.wav'  # Output file
reduce_wind_noise(input_audio_file, output_audio_file)

In [ ]:
#claude sonnet 3.7

#!/usr/bin/env python3
import pyaudio
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import time
import threading
import queue
import argparse
from datetime import datetime
import os

class WindNoiseDetector:
    def __init__(self, threshold=0.5, sample_rate=44100, chunk_size=4096, 
                 wind_freq_range=(20, 200), recording_seconds=5, 
                 apply_filter=False, log_file=None):
        self.threshold = threshold
        self.sample_rate = sample_rate
        self.chunk_size = chunk_size
        self.wind_freq_range = wind_freq_range
        self.recording_seconds = recording_seconds
        self.apply_filter = apply_filter
        self.log_file = log_file
        
        # Create a queue for audio data
        self.audio_queue = queue.Queue()
        self.running = False
        
        # Set up PyAudio
        self.p = pyaudio.PyAudio()
        
        # For visualization
        self.wind_scores = []
        self.timestamps = []
        
    def _callback(self, in_data, frame_count, time_info, status):
        """Callback function for PyAudio"""
        audio_data = np.frombuffer(in_data, dtype=np.float32)
        self.audio_queue.put(audio_data)
        return (in_data, pyaudio.paContinue)
    
    def _apply_highpass_filter(self, data, cutoff=200):
        """Apply a high-pass filter to reduce wind noise"""
        b, a = signal.butter(4, cutoff / (self.sample_rate / 2), 'highpass')
        return signal.filtfilt(b, a, data)
    
    def _calculate_wind_score(self, audio_data):
        """Calculate a 'wind score' based on low frequency energy"""
        # Calculate frequency spectrum
        fft_data = np.abs(np.fft.rfft(audio_data))
        freqs = np.fft.rfftfreq(len(audio_data), 1/self.sample_rate)
        
        # Isolate the wind frequency range
        wind_mask = (freqs >= self.wind_freq_range[0]) & (freqs <= self.wind_freq_range[1])
        non_wind_mask = (freqs > self.wind_freq_range[1]) & (freqs < 1000)  # Reference range
        
        # Calculate energy in wind range and non-wind range
        wind_energy = np.sum(fft_data[wind_mask])
        non_wind_energy = np.sum(fft_data[non_wind_mask])
        
        # Normalize by the number of frequency bins
        wind_energy = wind_energy / np.sum(wind_mask)
        non_wind_energy = non_wind_energy / np.sum(non_wind_mask)
        
        # Wind score is the ratio of wind energy to non-wind energy
        if non_wind_energy > 0:
            wind_score = wind_energy / non_wind_energy
        else:
            wind_score = 0
            
        return wind_score
    
    def _process_audio(self):
        """Process audio data from the queue"""
        while self.running:
            try:
                audio_data = self.audio_queue.get(timeout=1)
                
                # Calculate wind score
                wind_score = self._calculate_wind_score(audio_data)
                
                # Apply filter if requested
                if self.apply_filter and wind_score > self.threshold:
                    filtered_data = self._apply_highpass_filter(audio_data)
                    # Here you could output the filtered audio if needed
                
                # Log the results
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self.wind_scores.append(wind_score)
                self.timestamps.append(time.time())
                
                # Print status
                status = "HIGH WIND DETECTED!" if wind_score > self.threshold else "Normal"
                print(f"[{timestamp}] Wind score: {wind_score:.2f} - {status}")
                
                # Log to file if specified
                if self.log_file:
                    with open(self.log_file, 'a') as f:
                        f.write(f"{timestamp},{wind_score:.4f},{status}\n")
                        
                self.audio_queue.task_done()
            except queue.Empty:
                continue
            except Exception as e:
                print(f"Error processing audio: {e}")
    
    def start(self):
        """Start monitoring for wind noise"""
        if self.running:
            print("Already running")
            return
            
        # Create log file if needed
        if self.log_file:
            log_dir = os.path.dirname(self.log_file)
            if log_dir and not os.path.exists(log_dir):
                os.makedirs(log_dir)
            with open(self.log_file, 'w') as f:
                f.write("timestamp,wind_score,status\n")
        
        # Start the stream
        self.stream = self.p.open(
            format=pyaudio.paFloat32,
            channels=1,
            rate=self.sample_rate,
            input=True,
            frames_per_buffer=self.chunk_size,
            stream_callback=self._callback
        )
        
        # Start processing thread
        self.running = True
        self.process_thread = threading.Thread(target=self._process_audio)
        self.process_thread.daemon = True
        self.process_thread.start()
        
        print(f"Started wind noise detection (threshold: {self.threshold})")
        print(f"Monitoring frequencies: {self.wind_freq_range[0]}-{self.wind_freq_range[1]} Hz")
        if self.apply_filter:
            print("High-pass filter enabled")
        if self.log_file:
            print(f"Logging to: {self.log_file}")
        
        # Run for specified amount of time
        if self.recording_seconds > 0:
            time.sleep(self.recording_seconds)
            self.stop()
            self.visualize_results()
    
    def stop(self):
        """Stop monitoring"""
        if not self.running:
            return
            
        self.running = False
        self.stream.stop_stream()
        self.stream.close()
        if hasattr(self, 'process_thread'):
            self.process_thread.join(timeout=1)
        print("Stopped wind noise detection")
    
    def cleanup(self):
        """Clean up resources"""
        self.stop()
        self.p.terminate()
    
    def visualize_results(self):
        """Visualize the wind scores over time"""
        if not self.wind_scores:
            print("No data to visualize")
            return
            
        # Convert to relative time in seconds
        rel_times = [t - self.timestamps[0] for t in self.timestamps]
        
        plt.figure(figsize=(10, 6))
        plt.plot(rel_times, self.wind_scores)
        plt.axhline(y=self.threshold, color='r', linestyle='--', label=f'Threshold ({self.threshold})')
        plt.fill_between(rel_times, self.wind_scores, 0, 
                         where=[score > self.threshold for score in self.wind_scores],
                         color='red', alpha=0.3)
        plt.xlabel('Time (seconds)')
        plt.ylabel('Wind Score')
        plt.title('Wind Noise Detection')
        plt.grid(True)
        plt.legend()
        
        # Save the plot if logging is enabled
        if self.log_file:
            plot_file = self.log_file.replace('.csv', '.png')
            plt.savefig(plot_file)
            print(f"Plot saved to: {plot_file}")
        
        plt.show()


def main():
    parser = argparse.ArgumentParser(description='Detect and measure wind noise from a microphone')
    parser.add_argument('--threshold', type=float, default=0.5, 
                        help='Threshold for wind detection (default: 0.5)')
    parser.add_argument('--sample-rate', type=int, default=44100, 
                        help='Audio sample rate (default: 44100 Hz)')
    parser.add_argument('--chunk-size', type=int, default=4096, 
                        help='Audio chunk size (default: 4096)')
    parser.add_argument('--min-freq', type=int, default=20, 
                        help='Minimum frequency for wind detection (default: 20 Hz)')
    parser.add_argument('--max-freq', type=int, default=200, 
                        help='Maximum frequency for wind detection (default: 200 Hz)')
    parser.add_argument('--time', type=int, default=30, 
                        help='Recording time in seconds (default: 30, 0 for continuous)')
    parser.add_argument('--filter', action='store_true', 
                        help='Apply high-pass filter to reduce wind noise')
    parser.add_argument('--log', type=str, default=None, 
                        help='Log file path (CSV format)')
    
    args = parser.parse_args()
    
    detector = WindNoiseDetector(
        threshold=args.threshold,
        sample_rate=args.sample_rate,
        chunk_size=args.chunk_size,
        wind_freq_range=(args.min_freq, args.max_freq),
        recording_seconds=args.time,
        apply_filter=args.filter,
        log_file=args.log
    )
    
    try:
        detector.start()
    except KeyboardInterrupt:
        print("\nInterrupted by user")
    finally:
        detector.cleanup()


if __name__ == "__main__":
    main()

In [ ]:
#perplexity
# Example using noisereduce (requires noise sample)
import noisereduce as nr
import numpy as np

# Capture noise profile during "silent" wind-only moments
# noise_sample = np.frombuffer(audio_stream.read(CHUNK), dtype=np.float32)

# Apply noise reduction to live audio
denoised_audio = nr.reduce_noise(audio_clip=live_audio, noise_clip=noise_sample)



In [ ]:
#perplexity 2
import numpy as np
from scipy.fft import fft

def detect_wind(audio_data, sample_rate, threshold_db=30):
    spectrum = np.abs(fft(audio_data))
    freqs = np.fft.fftfreq(len(spectrum), 1/sample_rate)
    low_freq_mask = (freqs > 20) & (freqs < 500)  # Wind-dominated range
    wind_energy = np.sum(spectrum[low_freq_mask])
    total_energy = np.sum(spectrum)
    wind_ratio = wind_energy / total_energy
    return wind_ratio > (10**(-threshold_db/10))  # Threshold in dB


In [ ]:
import numpy as np
#Copilot
import sounddevice as sd

import matplotlib.pyplot as plt

# Configuration
DURATION = 5  # seconds
SAMPLERATE = 44100  # Hz
WIND_FREQ_RANGE = (20, 300)  # Hz
THRESHOLD_RATIO = 0.3  # Threshold for wind noise detection

def record_audio(duration, samplerate):
    """Record audio from the microphone."""
    print("Recording...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float64')
    sd.wait()
    print("Recording complete.")
    return audio.flatten()

def analyze_wind_noise(audio, samplerate, wind_freq_range, threshold_ratio):
    """Analyze the audio for wind noise."""
    # Perform FFT
    fft_vals = np.abs(np.fft.rfft(audio))
    fft_freqs = np.fft.rfftfreq(len(audio), 1 / samplerate)
    
    # Calculate energy in the wind frequency range
    wind_mask = (fft_freqs >= wind_freq_range[0]) & (fft_freqs <= wind_freq_range[1])
    wind_energy = np.sum(fft_vals[wind_mask])
    total_energy = np.sum(fft_vals)
    
    # Calculate the ratio of wind energy to total energy
    wind_ratio = wind_energy / total_energy
    print(f"Wind energy ratio: {wind_ratio:.2f}")
    
    # Determine if wind noise is significant
    if wind_ratio > threshold_ratio:
        print("⚠️ Significant wind noise detected!")
    else:
        print("✅ No significant wind noise detected.")
    
    # Plot the frequency spectrum
    plt.plot(fft_freqs, fft_vals)
    plt.title("Frequency Spectrum")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.xlim(0, 1000)  # Focus on low frequencies
    plt.show()

# Main script
if __name__ == "__main__":
    audio = record_audio(DURATION, SAMPLERATE)
    analyze_wind_noise(audio, SAMPLERATE, WIND_FREQ_RANGE, THRESHOLD_RATIO)

In [ ]:
import numpy as np
import pyaudio
import scipy.signal as signal
from scipy.io import wavfile
import matplotlib.pyplot as plt

class WindNoiseDetector:
    def __init__(self):
        # Audio parameters
        self.FORMAT = pyaudio.paFloat32
        self.CHANNELS = 1
        self.RATE = 44100
        self.CHUNK = 4096
        self.THRESHOLD = 0.1  # Adjust this threshold based on your needs
        
        # Wind noise typically occurs in low frequencies (20-200 Hz)
        self.WIND_FREQ_LOW = 20
        self.WIND_FREQ_HIGH = 200
        
        # Initialize PyAudio
        self.p = pyaudio.PyAudio()
        
    def start_monitoring(self, duration_seconds=10):
        try:
            # Open stream
            stream = self.p.open(
                format=self.FORMAT,
                channels=self.CHANNELS,
                rate=self.RATE,
                input=True,
                frames_per_buffer=self.CHUNK
            )
            
            print("Starting wind noise monitoring...")
            
            # Calculate number of chunks to read
            chunks_to_read = int((self.RATE * duration_seconds) / self.CHUNK)
            
            for _ in range(chunks_to_read):
                # Read audio data
                data = np.frombuffer(stream.read(self.CHUNK), dtype=np.float32)
                
                # Calculate wind noise metrics
                wind_level = self.measure_wind_noise(data)
                
                # Print wind level
                self.display_wind_level(wind_level)
                
        except Exception as e:
            print(f"Error: {e}")
        finally:
            # Clean up
            stream.stop_stream()
            stream.close()
            
    def measure_wind_noise(self, audio_data):
        # Apply bandpass filter to isolate wind frequency range
        nyquist = self.RATE // 2
        low = self.WIND_FREQ_LOW / nyquist
        high = self.WIND_FREQ_HIGH / nyquist
        b, a = signal.butter(4, [low, high], btype='band')
        filtered_data = signal.filtfilt(b, a, audio_data)
        
        # Calculate RMS of the filtered signal
        wind_level = np.sqrt(np.mean(filtered_data**2))
        
        return wind_level
    
    def display_wind_level(self, level):
        # Convert level to dB
        db_level = 20 * np.log10(level + 1e-10)  # Adding small number to avoid log(0)
        
        # Create a simple visual representation
        if db_level < -60:
            status = "No wind"
        elif db_level < -40:
            status = "Light wind"
        elif db_level < -20:
            status = "Moderate wind"
        else:
            status = "Strong wind"
            
        print(f"Wind Level: {db_level:.1f} dB - {status}")
    
    def close(self):
        self.p.terminate()

if __name__ == "__main__":
    detector = WindNoiseDetector()
    try:
        detector.start_monitoring(duration_seconds=30)  # Monitor for 30 seconds
    finally:
        detector.close()

In [ ]:
import sounddevice as sd
import numpy as np
import time

# --- Configuration ---
SAMPLE_RATE = 44100       # Sample rate in Hz
CHUNK_DURATION_S = 0.5    # Process audio in chunks of this duration
LOW_FREQ_THRESHOLD_HZ = 150 # Upper limit for "low frequency" band

# Calculate chunk size
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION_S)

# --- Callback Function ---
def audio_callback(indata, frames, time_info, status):
    """This function is called whenever a new audio chunk is available."""
    if status:
        print(status)

    # Calculate FFT
    # Use np.hanning(frames) window? Optional, can reduce spectral leakage
    fft_data = np.fft.rfft(indata[:, 0]) # Use only one channel (mono)
    fft_freq = np.fft.rfftfreq(frames, 1.0 / SAMPLE_RATE)

    # Find magnitude (amplitude) - absolute value of complex FFT result
    fft_magnitude = np.abs(fft_data)

    # Find energy/power in the low-frequency band
    low_freq_indices = np.where(fft_freq <= LOW_FREQ_THRESHOLD_HZ)[0]

    if len(low_freq_indices) > 0:
        # Sum magnitude squared for power, or just magnitude
        # Using mean magnitude here for simplicity
        low_freq_energy = np.mean(fft_magnitude[low_freq_indices])
    else:
        low_freq_energy = 0

    # --- Do something with the measurement ---
    # Simple print for demonstration
    # In a real application: compare to threshold, log data, trigger action, etc.
    print(f"Low Frequency Energy (~0-{LOW_FREQ_THRESHOLD_HZ} Hz): {low_freq_energy:.2f}")

    # Optional: Calculate overall RMS for comparison
    # rms_val = np.sqrt(np.mean(indata**2))
    # print(f"Overall RMS: {rms_val:.4f}")


# --- Main Execution ---
print("Starting audio stream...")
print(f"Chunk size: {CHUNK_SAMPLES} samples ({CHUNK_DURATION_S} seconds)")
print(f"Monitoring energy below {LOW_FREQ_THRESHOLD_HZ} Hz")

try:
    # Start the audio stream
    with sd.InputStream(samplerate=SAMPLE_RATE,
                       channels=1, # Use 1 channel (mono)
                       blocksize=CHUNK_SAMPLES,
                       callback=audio_callback):
        print("Stream started. Press Ctrl+C to stop.")
        # Keep the script running while the stream is active
        while True:
            time.sleep(0.1)

except KeyboardInterrupt:
    print("\nStopping stream.")
except Exception as e:
    print(f"An error occurred: {e}")